In [1]:
import os
from pathlib import Path

import pandas as pd
from googleapiclient.discovery import build


def _load_env_file(path: str = ".env") -> None:
    p = Path(path)
    if not p.exists():
        return

    for raw_line in p.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key:
            os.environ.setdefault(key, value)


_load_env_file()


In [2]:
API_KEY = os.getenv("YOUTUBE_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "Missing YOUTUBE_API_KEY. Create a .env file (see .env.example) or set it in your environment."
    )

In [3]:
youtube = build("youtube", "v3", developerKey=API_KEY)


In [4]:
def get_channel_id(channel_name):
    request = youtube.search().list(
        q=channel_name,
        part="snippet",
        type="channel",
        maxResults=1
    )
    response = request.execute()
    return response["items"][0]["snippet"]["channelId"]


In [5]:
def get_channel_stats(channel_id):
    request = youtube.channels().list(
        part="statistics",
        id=channel_id
    )
    response = request.execute()
    stats = response["items"][0]["statistics"]
    return int(stats["subscriberCount"]), int(stats["viewCount"])


In [6]:
def get_recent_videos(channel_id, max_results=10):
    request = youtube.search().list(
        channelId=channel_id,
        part="snippet",
        type="video",
        order="date",
        maxResults=max_results
    )
    response = request.execute()
    return [item["id"]["videoId"] for item in response["items"]]


In [7]:
def get_video_engagement(video_ids):
    likes, comments = [], []

    for vid in video_ids:
        request = youtube.videos().list(
            part="statistics",
            id=vid
        )
        response = request.execute()
        stats = response["items"][0]["statistics"]

        likes.append(int(stats.get("likeCount", 0)))
        comments.append(int(stats.get("commentCount", 0)))

    return sum(likes) / len(likes), sum(comments) / len(comments)


In [8]:
df = pd.read_csv("youtube_channels_sample.csv")
df


,channel_url
0,https://www.youtube.com/@PhysicsWallah
1,https://www.youtube.com/@PW-Foundation
2,https://www.youtube.com/@PW-JEEWallah
3,https://www.youtube.com/@PW-NEETWallah
4,https://www.youtube.com/@pwmeded
5,https://www.youtube.com/@englishpw
6,https://www.youtube.com/@AlakhSir-Class9.10
7,https://www.youtube.com/@vidyapeethpw
8,https://www.youtube.com/@PWLittleChamps


In [9]:
results = []

for url in df["channel_url"]:
    channel_name = url.split("@")[-1]

    channel_id = get_channel_id(channel_name)
    subscribers, total_views = get_channel_stats(channel_id)

    video_ids = get_recent_videos(channel_id)
    avg_likes, avg_comments = get_video_engagement(video_ids)

    engagement_rate = ((avg_likes + avg_comments) / subscribers) * 100

    results.append({
        "Channel Name": channel_name,
        "Subscribers": subscribers,
        "Total Views": total_views,
        "Avg Likes (Last 10 Videos)": int(avg_likes),
        "Avg Comments (Last 10 Videos)": int(avg_comments),
        "Engagement Rate (%)": round(engagement_rate, 3)
    })

results


[{'Channel Name': 'PhysicsWallah',
  'Subscribers': 14000000,
  'Total Views': 3065605012,
  'Avg Likes (Last 10 Videos)': 362,
  'Avg Comments (Last 10 Videos)': 0,
  'Engagement Rate (%)': 0.003},
 {'Channel Name': 'PW-Foundation',
  'Subscribers': 6210000,
  'Total Views': 2066288234,
  'Avg Likes (Last 10 Videos)': 14951,
  'Avg Comments (Last 10 Videos)': 892,
  'Engagement Rate (%)': 0.255},
 {'Channel Name': 'PW-JEEWallah',
  'Subscribers': 3090000,
  'Total Views': 1634933544,
  'Avg Likes (Last 10 Videos)': 7209,
  'Avg Comments (Last 10 Videos)': 148,
  'Engagement Rate (%)': 0.238},
 {'Channel Name': 'PW-NEETWallah',
  'Subscribers': 4470000,
  'Total Views': 2123511076,
  'Avg Likes (Last 10 Videos)': 9456,
  'Avg Comments (Last 10 Videos)': 316,
  'Engagement Rate (%)': 0.219},
 {'Channel Name': 'pwmeded',
  'Subscribers': 306000,
  'Total Views': 35043716,
  'Avg Likes (Last 10 Videos)': 1785,
  'Avg Comments (Last 10 Videos)': 7,
  'Engagement Rate (%)': 0.586},
 {'Chann

In [10]:
output_df = pd.DataFrame(results)
output_df = output_df.sort_values("Engagement Rate (%)", ascending=False)
output_df["Rank"] = range(1, len(output_df) + 1)
output_df


,Channel Name,Subscribers,Total Views,Avg Likes (Last 10 Videos),Avg Comments (Last 10 Videos),Engagement Rate (%),Rank
6,AlakhSir-Class9.10,1930000,322812911,78331,3913,4.261,1
4,pwmeded,306000,35043716,1785,7,0.586,2
1,PW-Foundation,6210000,2066288234,14951,892,0.255,3
2,PW-JEEWallah,3090000,1634933544,7209,148,0.238,4
3,PW-NEETWallah,4470000,2123511076,9456,316,0.219,5
8,PWLittleChamps,1110000,175805437,1697,646,0.211,6
7,vidyapeethpw,1000000,745458273,689,8,0.070,7
5,englishpw,680000,60770046,26,1,0.004,8
0,PhysicsWallah,14000000,3065605012,362,0,0.003,9


In [11]:
output_df.to_csv("youtube_analysis_output.csv", index=False)
